# Phase 1G: Preprocessing Validation

This notebook validates the processed dataset produced by the preprocessing pipeline.

**Important:** This notebook does not modify raw data.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def resolve_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "soil_ai_system").exists():
        return cwd / "soil_ai_system"
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd


project_root = resolve_project_root()
sys.path.append(str(project_root))

from config import (
    PROCESSED_DATA_PATH,
    PROCESSED_DATASETS,
    REPORT_PATH,
    PIPELINE_CONFIGS,
    N_MIN,
    N_MAX,
    P_MIN,
    P_MAX,
    K_MIN,
    K_MAX,
    PH_MIN_ALLOWED,
    PH_MAX_ALLOWED,
    MOISTURE_MIN,
    MOISTURE_MAX,
    HUMIDITY_MIN,
    HUMIDITY_MAX,
    RAINFALL_MIN,
    RAINFALL_MAX,
    TEMPERATURE_MIN,
    TEMPERATURE_MAX,
)

report_dir = (project_root / REPORT_PATH / "figures").resolve()
report_dir.mkdir(parents=True, exist_ok=True)

# Define range checks for validation
RANGE_CHECKS = {
    "N": (N_MIN, N_MAX),
    "P": (P_MIN, P_MAX),
    "K": (K_MIN, K_MAX),
    "ph": (PH_MIN_ALLOWED, PH_MAX_ALLOWED),
    "moisture": (MOISTURE_MIN, MOISTURE_MAX),
    "humidity": (HUMIDITY_MIN, HUMIDITY_MAX),
    "rainfall": (RAINFALL_MIN, RAINFALL_MAX),
    "temperature": (TEMPERATURE_MIN, TEMPERATURE_MAX),
}

print(f"Reports directory: {report_dir}")
print("Available processed datasets:", list(PROCESSED_DATASETS.keys()))

In [ ]:
dataset_dfs = {}
for key, filename in PROCESSED_DATASETS.items():
    processed_path = (project_root / PROCESSED_DATA_PATH / filename).resolve()
    if processed_path.exists():
        dataset_dfs[key] = pd.read_csv(processed_path)
        print(f"Loaded {key}: {dataset_dfs[key].shape}")
    else:
        print(f"Dataset {key} not found at {processed_path}")

# Default to first available for subsequent cells
if dataset_dfs:
    current_key = list(dataset_dfs.keys())[0]
    df = dataset_dfs[current_key]
    print(f"\nDefaulting to dataset: {current_key}")
else:
    print("\nNo datasets found!")

In [ ]:
for key, df_item in dataset_dfs.items():
    print(f"\n--- Validation Summary: {key} ---")
    config = PIPELINE_CONFIGS.get(key, {})
    
    missing_features = [col for col in config.get("features", []) if col not in df_item.columns]
    target = config.get("target")
    missing_target = [target] if target and target not in df_item.columns else []
    
    print(f"Missing feature columns: {missing_features}")
    print(f"Missing target column: {missing_target}")
    
    null_counts = df_item.isnull().sum()
    if null_counts.sum() > 0:
        print("Null values found:")
        print(null_counts[null_counts > 0])
    else:
        print("No null values.")
        
    print(f"Duplicate rows: {df_item.duplicated().sum()}")

In [ ]:
for key, df_item in dataset_dfs.items():
    print(f"\n--- Range Validation: {key} ---")
    range_results = []
    
    for col, (lower, upper) in RANGE_CHECKS.items():
        if col in df_item.columns:
            c_min = df_item[col].min()
            c_max = df_item[col].max()
            valid = (c_min >= lower - 0.01) and (c_max <= upper + 0.01) # Small epsilon for scaling
            range_results.append({
                "Feature": col,
                "Min": f"{c_min:.2f}",
                "Max": f"{c_max:.2f}",
                "Allowed": f"[{lower}, {upper}]",
                "Status": "PASS" if valid else "CLIP/FAIL"
            })
            
    pd.set_option('display.max_columns', None)
    results_df = pd.DataFrame(range_results)
    display(results_df)

In [ ]:
# Distribution visualization across all datasets
for key, df_item in dataset_dfs.items():
    plt.figure(figsize=(15, 5))
    cols_to_plot = [c for c in ["N", "P", "K", "ph"] if c in df_item.columns]
    
    if not cols_to_plot:
        continue
        
    for i, col in enumerate(cols_to_plot):
        plt.subplot(1, len(cols_to_plot), i+1)
        sns.histplot(df_item[col], kde=True)
        plt.title(f"{key}: {col}")
        
    plt.tight_layout()
    plt.savefig(report_dir / f"distributions_{key}.png")
    plt.show()

In [ ]:
for col in TARGET_COLS:
    if col not in df.columns:
        continue
    plt.figure(figsize=(6, 4))
    df[col].value_counts(dropna=False).plot(kind="bar")
    plt.title(f"Class Balance: {col}")
    plt.tight_layout()
    plot_path = report_dir / f"class_balance_{col}.png"
    plt.savefig(plot_path)
    plt.close()
    print(f"Saved {plot_path}")